# 05 · Validação cruzada e generalização (T1)

**Onde estamos no ML:** entregável 6. Pergunta: os resultados dos dois melhores modelos (Regressão Logística e Random Forest otimizado) são **estáveis**, mostram **sobreajuste ou subajuste** e **generalizam** para dados diferentes? Tudo está no MLflow (experimento `t1-manter-descartar`).

| Análise | O que mede | Como |
| --- | --- | --- |
| Validação cruzada | estabilidade dos resultados | `StratifiedGroupKFold`, k=5, agrupada por frase |
| Treino × validação | sobreajuste / subajuste | F1-macro de treino contra o da dobra |
| Curva de aprendizado | se mais dados ajudariam | treino em 2% a 100% das frases |
| Fora do domínio | generalização | treina em geral e testa em governo, e vice-versa |

**Cuidados.** A validação cruzada usa uma amostra de **25.000 frases** (~320 mil palavras) do conjunto de treino, porque 5 florestas completas levariam horas; os valores absolutos são por isso um pouco menores que os do modelo treinado em tudo (~1,3 milhão de palavras). O teste final (10% das frases) **não** foi usado em nenhuma destas análises.

In [1]:
import os, sys, json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import mlflow
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from textgloss import train
URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(URI); mlflow.set_experiment(train.EXPERIMENT)
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name(train.EXPERIMENT).experiment_id
def run(nome):
    return client.search_runs([experiment], filter_string=f"tags.modelo = '{nome}' and attributes.status = 'FINISHED'", order_by=["attributes.start_time DESC"])[0]
def tabela(run_id, arquivo):
    caminho = mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{run_id}/{arquivo}", tracking_uri=URI)
    dados = json.load(open(caminho)); return pd.DataFrame(dados["data"], columns=dados["columns"])
def mostrar(run_id, caminho, tamanho=(8, 4.5)):
    arquivo = mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{run_id}/{caminho}", tracking_uri=URI)
    plt.figure(figsize=tamanho); plt.imshow(mpimg.imread(arquivo)); plt.axis("off"); plt.tight_layout()
cv, curva, dominio = run("validacao-cruzada"), run("curva-aprendizado"), run("fora-do-dominio")
print("MLflow:", URI)

MLflow: http://localhost:5000


## 1. Validação cruzada (k=5, agrupada por frase)
As palavras de uma mesma frase ficam sempre na mesma dobra; sem isso o modelo veria no treino as palavras vizinhas das que vai prever. Os dois modelos usam exatamente as mesmas dobras.

In [2]:
dobras = tabela(cv.info.run_id, "dobras.json")
resumo = dobras.groupby("modelo").agg(f1_media=("f1_macro_val", "mean"), f1_desvio=("f1_macro_val", "std"), f1_min=("f1_macro_val", "min"),
                                      f1_max=("f1_macro_val", "max"), roc_auc_media=("roc_auc_val", "mean"), pr_auc_media=("pr_auc_val", "mean"),
                                      rec_descartar=("recall_descartar_val", "mean"), treino_medio_s=("tempo_treino_s", "mean"))
resumo["cv_%"] = 100 * resumo.f1_desvio / resumo.f1_media
resumo.round(4)

,f1_media,f1_desvio,f1_min,f1_max,roc_auc_media,pr_auc_media,rec_descartar,treino_medio_s,cv_%
modelo,,,,,,,,,
random-forest-otimizado,0.8789,0.0013,0.8768,0.8804,0.9425,0.9348,0.8241,95.1431,0.1534
regressao-logistica,0.8747,0.0014,0.8731,0.8759,0.9426,0.9400,0.8191,5.8136,0.1579


In [3]:
mostrar(cv.info.run_id, "cv_por_dobra.png", (7, 4))
pares = dobras.pivot(index="dobra", columns="modelo", values="f1_macro_val")
pares["diferenca_RF_menos_LR"] = pares["random-forest-otimizado"] - pares["regressao-logistica"]
print(f"Random Forest melhor que a Regressão em {(pares.diferenca_RF_menos_LR > 0).sum()} de {len(pares)} dobras")
print(f"diferença média {pares.diferenca_RF_menos_LR.mean():.4f} · desvio {pares.diferenca_RF_menos_LR.std(ddof=1):.4f}")
pares.round(4)

Random Forest melhor que a Regressão em 5 de 5 dobras
diferença média 0.0043 · desvio 0.0008


modelo,random-forest-otimizado,regressao-logistica,diferenca_RF_menos_LR
dobra,,,
0,0.8792,0.8753,0.0040
1,0.8786,0.8731,0.0055
2,0.8795,0.8759,0.0036
3,0.8768,0.8732,0.0036
4,0.8804,0.8758,0.0046


## 2. Sobreajuste ou subajuste: treino contra validação, por dobra

In [4]:
dobras["gap"] = dobras.f1_macro_treino - dobras.f1_macro_val
dobras.groupby("modelo").agg(f1_treino=("f1_macro_treino", "mean"), f1_validacao=("f1_macro_val", "mean"), gap_medio=("gap", "mean"), gap_max=("gap", "max")).round(4)

,f1_treino,f1_validacao,gap_medio,gap_max
modelo,,,,
random-forest-otimizado,0.9010,0.8789,0.0221,0.0246
regressao-logistica,0.8763,0.8747,0.0016,0.0037


## 3. Curva de aprendizado
Treino em frações crescentes das frases de ajuste (2%, 5%, 10%, 25%, 50%; o ponto de 100% vem das runs anteriores), avaliação sempre na mesma validação.

In [5]:
mostrar(curva.info.run_id, "curva_aprendizado.png", (11, 4))
c = tabela(curva.info.run_id, "curva.json"); c["gap"] = c.f1_macro_treino - c.f1_macro_val
c.round(4)

,modelo,fracao,n_palavras,f1_macro_treino,f1_macro_val,tempo_treino_s,gap
0,regressao-logistica,0.02,25860,0.8796,0.8657,0.3827,0.0139
1,regressao-logistica,0.05,64893,0.8784,0.8712,1.1101,0.0071
2,regressao-logistica,0.10,130114,0.8788,0.8740,2.3655,0.0048
3,regressao-logistica,0.25,328102,0.8758,0.8753,7.6550,0.0005
4,regressao-logistica,0.50,653237,0.8759,0.8753,14.7529,0.0006
5,regressao-logistica,1.00,1308450,0.8770,0.8759,40.8371,0.0011
6,random-forest-otimizado,0.02,25860,0.8999,0.8652,2.3443,0.0347
7,random-forest-otimizado,0.05,64893,0.9020,0.8727,7.7826,0.0293
8,random-forest-otimizado,0.10,130114,0.9024,0.8766,25.3542,0.0258
9,random-forest-otimizado,0.25,328102,0.9014,0.8808,157.3901,0.0205


## 4. Fora do domínio
O corpus tem dois domínios: **geral** (frases de coleções de exemplos e textos variados) e **governo** (textos do gov.br). Treinar em um e avaliar no outro mede quanto o modelo generaliza.

In [6]:
mostrar(dominio.info.run_id, "fora_do_dominio.png", (7.5, 4))
d = tabela(dominio.info.run_id, "dominio.json")
dentro = d[d.dentro_do_dominio].set_index(["modelo", "avaliado_em"])["f1_macro"]
fora = d[~d.dentro_do_dominio].set_index(["modelo", "avaliado_em"])["f1_macro"]
queda = pd.DataFrame({"treino_no_proprio_dominio": dentro, "treino_no_outro_dominio": fora}); queda["queda"] = queda.treino_no_proprio_dominio - queda.treino_no_outro_dominio
print("Avaliado em (linha) · treinado no próprio domínio × no outro:"); queda.round(4)

Avaliado em (linha) · treinado no próprio domínio × no outro:


treino_no_proprio_dominio  \
modelo                  avaliado_em                              
random-forest-otimizado geral                           0.8863   
                        governo                         0.8897   
regressao-logistica     geral                           0.8733   
                        governo                         0.8890   

                                     treino_no_outro_dominio   queda  
modelo                  avaliado_em                                   
random-forest-otimizado geral                         0.8541  0.0323  
                        governo                       0.8476  0.0421  
regressao-logistica     geral                         0.8520  0.0213  
                        governo                       0.8576  0.0314

In [7]:
d.round(4)

,modelo,treino_em,avaliado_em,dentro_do_dominio,n_palavras_treino,f1_macro,roc_auc
0,regressao-logistica,geral,geral,True,655296,0.8733,0.9419
1,regressao-logistica,geral,governo,False,655296,0.8576,0.9150
2,regressao-logistica,governo,geral,False,653154,0.8520,0.9078
3,regressao-logistica,governo,governo,True,653154,0.8890,0.9566
4,random-forest-otimizado,geral,geral,True,655296,0.8863,0.9486
5,random-forest-otimizado,geral,governo,False,655296,0.8476,0.9115
6,random-forest-otimizado,governo,geral,False,653154,0.8541,0.9092
7,random-forest-otimizado,governo,governo,True,653154,0.8897,0.9527


## 5. Análise

**1. Os resultados são estáveis.** Na validação cruzada (k=5, agrupada por frase) o F1-macro varia pouco entre dobras: Random Forest otimizado **0,8789 ± 0,0013** (mín. 0,8768, máx. 0,8804) e Regressão Logística **0,8747 ± 0,0014** (0,8731 a 0,8759). O desvio é de ~0,15% da média nos dois. Nenhuma dobra destoa, então os números dos experimentos anteriores (uma validação só) não eram sorte de sorteio.

**2. O Random Forest é melhor em F1, mas por pouco.** Ele vence em **5 das 5 dobras**, por 0,0043 em média (desvio 0,0008, de 0,0036 a 0,0055): a diferença é pequena e consistente, não ruído. Mas ROC-AUC é igual (0,9425 contra 0,9426) e a PR-AUC é maior na regressão (0,940 contra 0,935): na qualidade da ordenação por confiança não há vantagem; o ganho está na decisão com corte em 0,5 e na revocação de `DESCARTAR` (0,824 contra 0,819).

**3. Sobreajuste e subajuste.**
- **Regressão Logística: subajuste leve.** Treino 0,876 e validação 0,875 (diferença de 0,002). A curva de aprendizado **estaciona em 25% dos dados** (0,8753) e não sobe até 100% (0,8759): o modelo é simples demais para extrair mais, e dar mais dados não ajuda.
- **Random Forest otimizado: sobreajuste moderado e ainda ganhando com dados.** Treino ~0,90 e validação 0,879 na CV (diferença de 0,022). A diferença **diminui** com mais dados (de 0,035 com 2% para 0,016 com 100%) e a validação continua subindo (0,8808 → 0,8830 → 0,8840 dos 25% aos 100%). Mais dados, ou mais atributos, ainda beneficiariam este modelo, ao contrário da regressão.

**4. Generalização fora do domínio é o ponto fraco dos dois.** Treinar em um domínio e testar no outro derruba o F1-macro entre 2 e 4 pontos:

| Avaliado em | Regressão (próprio → outro domínio) | Random Forest (próprio → outro domínio) |
| --- | --- | --- |
| geral | 0,873 → 0,852 (−0,021) | 0,886 → 0,854 (−0,032) |
| governo | 0,889 → 0,858 (−0,031) | 0,890 → 0,848 (−0,042) |

A queda média é de 0,026 na regressão e 0,037 no Random Forest. A regressão é mais robusta (e ganha do Random Forest em governo quando treinada em geral: 0,858 contra 0,848). Dentro do domínio, o Random Forest só ganha claramente em geral; em governo empata (0,8897 contra 0,8890). Os dois domínios compartilham só 20,5% dos lemas (EDA), o que explica boa parte da queda. Isso importa porque o Sinaliza vai receber texto de qualquer assunto.

**5. Leitura conjunta: não há vencedor claro.** O Random Forest ganha 0,4 ponto de F1 na validação cruzada e ~0,8 ponto na validação completa; a regressão perde menos fora do domínio, treina mais de 35 vezes mais rápido (41 s contra 1.520 s), é explicável pelos coeficientes e não sobreajusta. A escolha depende do critério do grupo; qualquer das duas é defensável.

**6. Limites.**
- A CV usou uma amostra de 25 mil frases (~320 mil palavras); os valores absolutos são menores que os do modelo treinado em tudo.
- Os hiperparâmetros do Random Forest foram escolhidos pelo Optuna com frases do mesmo conjunto de treino de onde saiu a amostra da CV, então o Random Forest tem uma leve vantagem de ter sido ajustado "perto" desses dados. A regressão não foi otimizada.
- O ruído de rótulo (conferência manual pendente) limita o teto de todos os modelos.
- A "queda" fora do domínio mistura mudança de domínio com menos dados de treino (metade do total em cada caso).
- O **teste final ainda não foi usado**; vai ser rodado uma única vez, para o modelo escolhido.
- A avaliação ponta a ponta contra o `vlibras-translate` (BLEU/chrF/WER) não foi feita.
